# Trabajo Práctico Integrador

## Urban Flow


## Integrantes del grupo

- Completar con nombre y apellido.
- Completar con nombre y apellido.


## Sprint 1

Primer avance del proyecto: configuración del repositorio, estructura de carpetas y base del notebook para los próximos ejercicios.


## Configuración inicial

En esta primera entrega parcial se deja preparado el entorno de trabajo:

- Imports generales.
- Variables globales.
- Estructura de carpetas.
- Flujo básico de Git con `desactivar_git_push`.


In [ ]:
from urllib.request import urlretrieve
from datetime import datetime
from pathlib import Path
import re
import subprocess
import unicodedata

import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
desactivar_git_push = True
nombre_rama = "Sprint_1"
url_repositorio_https = ""
dataset_url = "https://raw.githubusercontent.com/HAD141/datasets/refs/heads/main/TrabajosPracticos/urban_flow/speeding_fines.csv"

repo_root = Path.cwd()
project_root = repo_root / "urban_flow"
raw_dir = project_root / "data" / "raw"
interim_dir = project_root / "data" / "interim"
processed_dir = project_root / "data" / "processed"
plots_dir = interim_dir / "plots"
raw_dataset_path = raw_dir / "speeding_fines.csv"


## Ejercicio 01

> Puntos: 1

Inicialización y configuración de la herramienta de versionado. Se trabaja sobre la rama `Sprint_1` y se crea la estructura pedida para el proyecto.


In [ ]:
for directory in [raw_dir, interim_dir, processed_dir, plots_dir]:
    directory.mkdir(parents=True, exist_ok=True)

for ruta in [project_root, raw_dir, interim_dir, processed_dir, plots_dir]:
    print(ruta.relative_to(repo_root))


In [ ]:
def ejecutar_git(args: list[str]) -> None:
    resultado = subprocess.run(["git", *args], capture_output=True, text=True)
    if resultado.stdout.strip():
        print(resultado.stdout.strip())
    if resultado.stderr.strip():
        print(resultado.stderr.strip())

ramas = subprocess.run(["git", "branch", "--list", nombre_rama], capture_output=True, text=True)
if ramas.stdout.strip():
    ejecutar_git(["checkout", nombre_rama])
else:
    ejecutar_git(["checkout", "-b", nombre_rama])


In [ ]:
print("Estado actual del repositorio:")
ejecutar_git(["status", "--short", "--branch"])


In [ ]:
def obtener_github_token() -> str:
    try:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return ""

def push_si_corresponde() -> None:
    if desactivar_git_push:
        print("git push desactivado para esta ejecución.")
        return
    if not url_repositorio_https:
        print("No se configuró la URL del repositorio remoto.")
        return
    github_token = obtener_github_token()
    if not github_token:
        print("No se encontró GITHUB_TOKEN en los secrets de Colab.")
        return
    remote_url_con_token = url_repositorio_https.replace("https://", f"https://{github_token}@")
    subprocess.run(["git", "remote", "remove", "origin"], capture_output=True, text=True)
    subprocess.run(["git", "remote", "add", "origin", remote_url_con_token], capture_output=True, text=True)
    ejecutar_git(["push", "-u", "origin", nombre_rama])


In [ ]:
print("Comandos sugeridos para este primer avance:")
print("git add README.md")
print("git add CHANGELOG.md")
print("git add 01_Urban_Flow_Apellidos_Nombres.ipynb")
print("git add .gitignore")
print("git commit -m \"Sprint 1 - estructura inicial\"")


In [ ]:
push_si_corresponde()


## Ejercicio 02

> Puntos: 1

Se descarga el dataset original, se almacena en `urban_flow/data/raw`, y luego se inspeccionan las primeras filas, los tipos de datos y los valores nulos.


In [ ]:
urlretrieve(dataset_url, raw_dataset_path)
print(f"Dataset descargado en: {raw_dataset_path}")


In [ ]:
df_raw = pd.read_csv(raw_dataset_path)
df_raw.head()


In [ ]:
df_raw.dtypes


In [ ]:
df_raw.isna().sum()


In [ ]:
push_si_corresponde()


## Ejercicio 03

> Puntos: 2

Limpieza y transformación del dataset: normalización de fechas, horas, ubicaciones y patentes; eliminación de filas inválidas y outliers; cálculo de excesos de velocidad; y exportación del dataset limpio a `data/interim/`.


In [ ]:
FECHA_POR_DEFECTO = "1932-01-01"
HORA_POR_DEFECTO = "00:00"

def normalizar_texto(texto: object) -> str:
    if pd.isna(texto):
        return ""
    texto = str(texto).strip().upper()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(caracter for caracter in texto if not unicodedata.combining(caracter))
    texto = re.sub(r"[^A-Z0-9 ]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

def normalizar_fecha(valor: object) -> str:
    if pd.isna(valor):
        return FECHA_POR_DEFECTO
    texto = str(valor).strip()
    for formato in ("%d/%m/%Y", "%Y/%m/%d", "%Y-%m-%d"):
        try:
            return datetime.strptime(texto, formato).strftime("%Y-%m-%d")
        except ValueError:
            continue
    return FECHA_POR_DEFECTO

def normalizar_hora(valor: object) -> str:
    if pd.isna(valor):
        return HORA_POR_DEFECTO
    texto = str(valor).strip().upper()
    for formato in ("%I:%M %p", "%H:%M"):
        try:
            return datetime.strptime(texto, formato).strftime("%H:%M")
        except ValueError:
            continue
    return HORA_POR_DEFECTO

def normalizar_patente(valor: object):
    texto = normalizar_texto(valor).replace(" ", "")
    return texto if texto else pd.NA


In [ ]:
df_limpio = df_raw.copy()


In [ ]:
df_limpio["fecha"] = df_limpio["fecha"].apply(normalizar_fecha)
df_limpio["fecha"].head(10)


In [ ]:
df_limpio["hora"] = df_limpio["hora"].apply(normalizar_hora)
df_limpio["hora"].head(10)


In [ ]:
df_limpio["ubicacion"] = df_limpio["ubicacion"].apply(normalizar_texto)
df_limpio["ubicacion"].head(10)


In [ ]:
df_limpio["patente"] = df_limpio["patente"].apply(normalizar_patente)
df_limpio["patente"].tail(10)


### Eliminación de filas con valores relevantes vacíos

Se descartan las filas que no tienen los datos mínimos para considerarse una multa válida: `patente`, `velocidad_registrada` y `velocidad_maxima`.


In [ ]:
columnas_relevantes = ["patente", "velocidad_registrada", "velocidad_maxima"]

filas_antes = len(df_limpio)
df_limpio = df_limpio.dropna(subset=columnas_relevantes).reset_index(drop=True)
filas_eliminadas = filas_antes - len(df_limpio)

print(f"Se eliminaron {filas_eliminadas} filas.")
print("Las columnas observadas son:")
for columna in columnas_relevantes:
    print(f" - {columna}")


### Detección y eliminación de outliers

Se usa el rango intercuartílico (IQR) sobre las columnas numéricas de velocidad para descartar valores fuera de los límites razonables.


In [ ]:
def limites_iqr(serie: pd.Series) -> tuple[float, float]:
    q1, q3 = serie.quantile(0.25), serie.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

columnas_outliers = ["velocidad_registrada", "velocidad_maxima"]
mascara = pd.Series(False, index=df_limpio.index)
for columna in columnas_outliers:
    inferior, superior = limites_iqr(df_limpio[columna])
    mascara |= (df_limpio[columna] < inferior) | (df_limpio[columna] > superior)

filas_antes = len(df_limpio)
df_limpio = df_limpio.loc[~mascara].reset_index(drop=True)
filas_eliminadas = filas_antes - len(df_limpio)

print(f"Se eliminaron {filas_eliminadas} filas.")
print("Las columnas observadas son:")
for columna in columnas_outliers:
    print(f" - {columna}")


### Cálculo del exceso de velocidad real

Diferencia entre la velocidad registrada y la velocidad máxima permitida.


In [ ]:
df_limpio["exceso_velocidad_real"] = (
    df_limpio["velocidad_registrada"] - df_limpio["velocidad_maxima"]
)
df_limpio[["patente", "exceso_velocidad_real"]].head(10)


### Cálculo del exceso de velocidad con tolerancia

Se contempla un 5% de tolerancia sobre la velocidad máxima antes de considerar infracción.


In [ ]:
df_limpio["exceso_velocidad"] = (
    df_limpio["velocidad_registrada"] - df_limpio["velocidad_maxima"] * 1.05
)
df_limpio[["patente", "exceso_velocidad"]].head(10)


### Filtrado de filas sin infracción

Si `exceso_velocidad` no es positivo, la velocidad registrada queda dentro del margen tolerado y no corresponde multar.


In [ ]:
filas_antes = len(df_limpio)
df_limpio = df_limpio.loc[df_limpio["exceso_velocidad"] > 0].reset_index(drop=True)
filas_eliminadas = filas_antes - len(df_limpio)

print(f"Se eliminaron {filas_eliminadas} filas.")


### Exportación del dataset limpio

Se guarda el dataframe resultante en `urban_flow/data/interim/speeding_fines.csv` para que el próximo avance trabaje sobre datos ya depurados.


In [ ]:
interim_dataset_path = interim_dir / "speeding_fines.csv"
df_limpio.to_csv(interim_dataset_path, index=False)
print(f"Dataset limpio guardado en: {interim_dataset_path}")


In [ ]:
push_si_corresponde()


## Ejercicio 04

> Puntos: 2

Se define la clase `FineAnalyzer`, que encapsula el dataframe limpio y ofrece métodos para consultar rankings, promedios y totales por ubicación.


In [ ]:
class FineAnalyzer:
    """Analiza el dataframe limpio de multas por exceso de velocidad."""

    def __init__(self, df: pd.DataFrame) -> None:
        self._df = df.copy()

    def _ranking(self, columna: str, top: int = 5) -> pd.DataFrame:
        conteo = (
            self._df[columna]
            .value_counts()
            .head(top)
            .rename_axis(columna)
            .reset_index(name="cantidad")
        )
        conteo.index = range(1, len(conteo) + 1)
        return conteo

    def top_patentes_multadas(self) -> pd.DataFrame:
        """Top 5 patentes con mayor cantidad de multas."""
        return self._ranking("patente")

    def top_horarios_multas(self) -> pd.DataFrame:
        """Top 5 horarios con mayor cantidad de multas."""
        return self._ranking("hora")

    def exceso_velocidad_promedio(self) -> float:
        """Promedio del exceso de velocidad (con tolerancia del 5%)."""
        return float(self._df["exceso_velocidad"].mean())

    def exceso_velocidad_real_promedio(self) -> float:
        """Promedio del exceso de velocidad real (sin tolerancia)."""
        return float(self._df["exceso_velocidad_real"].mean())

    def multas_por_ubicacion(self) -> pd.DataFrame:
        """Cantidad de multas por ubicación, ordenadas por nombre."""
        conteo = (
            self._df.groupby("ubicacion")
            .size()
            .reset_index(name="cantidad")
            .sort_values("ubicacion")
            .reset_index(drop=True)
        )
        return conteo


### Instanciación del analizador

Se crea el objeto pasando el dataframe limpio obtenido en el Ejercicio 03.


In [ ]:
analizador = FineAnalyzer(df_limpio)


### Top 5 patentes más multadas


In [ ]:
analizador.top_patentes_multadas()


### Top 5 horarios con más multas


In [ ]:
analizador.top_horarios_multas()


### Exceso de velocidad promedio (con tolerancia)


In [ ]:
analizador.exceso_velocidad_promedio()


### Exceso de velocidad real promedio (sin tolerancia)


In [ ]:
analizador.exceso_velocidad_real_promedio()


### Cantidad de multas por ubicación


In [ ]:
analizador.multas_por_ubicacion()


In [ ]:
push_si_corresponde()


## Punto 05

> Puntos: 2

Se generan cinco gráficos y se exportan como `.jpg` dentro de `urban_flow/data/interim/plots/`. Para los dos últimos se aclara la interpretación usada.


### Top 10 patentes más reincidentes


In [ ]:
top_patentes = df_limpio["patente"].value_counts().head(10)

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(top_patentes.index, top_patentes.values, color="#4C72B0")
ax.set_xlabel("Patente")
ax.set_ylabel("Cantidad de multas")
ax.set_title("Top 10 patentes más reincidentes")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(plots_dir / "fines.jpg", format="jpg", dpi=100)
plt.show()


### Porcentaje de infracciones por hora

Se agrupa por la hora del reloj (0 a 23) en lugar de por el valor HH:MM completo para que la torta sea legible.


In [ ]:
por_hora = df_limpio["hora"].str.slice(0, 2).value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 10))
ax.pie(
    por_hora.values,
    labels=[f"{h}hs" for h in por_hora.index],
    autopct="%1.1f%%",
    startangle=90,
)
ax.set_title("Porcentaje de infracciones por hora")
plt.tight_layout()
plt.savefig(plots_dir / "hours.jpg", format="jpg", dpi=100)
plt.show()


### Cantidad de infracciones por mes

Se agrupa por mes del año (1 a 12) para que el gráfico tenga una escala acotada.


In [ ]:
nombres_meses = {
    1: "Ene", 2: "Feb", 3: "Mar", 4: "Abr", 5: "May", 6: "Jun",
    7: "Jul", 8: "Ago", 9: "Sep", 10: "Oct", 11: "Nov", 12: "Dic",
}

fechas = pd.to_datetime(df_limpio["fecha"], errors="coerce")
por_mes = fechas.dt.month.value_counts().sort_values()
etiquetas = [nombres_meses[m] for m in por_mes.index]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(etiquetas, por_mes.values, color="#55A868")
ax.set_xlabel("Cantidad de multas")
ax.set_ylabel("Mes")
ax.set_title("Cantidad de infracciones por mes")
plt.tight_layout()
plt.savefig(plots_dir / "months.jpg", format="jpg", dpi=100)
plt.show()


### Excesos de velocidad con hora 00:00

Interpretación: se filtran las filas cuya `hora` es `00:00` (valor por defecto para horas inválidas) y se grafica el exceso promedio a lo largo de la fecha. Se excluyen las filas que además tienen la fecha por defecto (`1932-01-01`) para evitar que ese único punto deforme la línea temporal.


In [ ]:
filtro_hora = df_limpio.loc[
    (df_limpio["hora"] == HORA_POR_DEFECTO)
    & (df_limpio["fecha"] != FECHA_POR_DEFECTO)
].copy()
filtro_hora["fecha_dt"] = pd.to_datetime(filtro_hora["fecha"], errors="coerce")
serie_hora = (
    filtro_hora.groupby("fecha_dt")["exceso_velocidad"].mean().sort_index()
)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(serie_hora.index, serie_hora.values, color="#C44E52")
ax.set_xlabel("Fecha")
ax.set_ylabel("Exceso de velocidad (km/h)")
ax.set_title("Excesos de velocidad en infracciones con hora 00:00")
fig.autofmt_xdate()
plt.tight_layout()
plt.savefig(plots_dir / "hour.jpg", format="jpg", dpi=100)
plt.show()


### Excesos de velocidad con fecha 1932-01-01

Interpretación: se filtran las filas cuya `fecha` es `1932-01-01` (valor por defecto para fechas inválidas), se excluyen además las que tienen hora por defecto, y se agrega el exceso promedio por hora del reloj (0–23) para que el eje X quede acotado y legible.


In [ ]:
filtro_fecha = df_limpio.loc[
    (df_limpio["fecha"] == FECHA_POR_DEFECTO)
    & (df_limpio["hora"] != HORA_POR_DEFECTO)
].copy()
filtro_fecha["hora_reloj"] = filtro_fecha["hora"].str.slice(0, 2)
serie_fecha = (
    filtro_fecha.groupby("hora_reloj")["exceso_velocidad"].mean().sort_index()
)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(serie_fecha.index, serie_fecha.values, color="#8172B2", marker="o")
ax.set_xlabel("Hora")
ax.set_ylabel("Exceso de velocidad (km/h)")
ax.set_title("Excesos de velocidad en infracciones con fecha 1932-01-01")
ax.set_xticks(range(len(serie_fecha.index)))
ax.set_xticklabels([f"{h}hs" for h in serie_fecha.index])
plt.tight_layout()
plt.savefig(plots_dir / "date.jpg", format="jpg", dpi=100)
plt.show()


In [ ]:
push_si_corresponde()


## Punto 06

> Puntos: 1

Se muestra qué porcentaje del total de multas cayó en los valores por defecto que se asignaron en el Ejercicio 03 (`1932-01-01` para fechas inválidas y `00:00` para horas inválidas), como diagnóstico de la calidad original del dato.


In [ ]:
porcentaje_fecha = (
    (df_limpio["fecha"] == FECHA_POR_DEFECTO).sum() / len(df_limpio) * 100
)
print(
    f"El porcentaje de infracciones en la fecha {FECHA_POR_DEFECTO} "
    f"es {porcentaje_fecha:.2f}%"
)


In [ ]:
porcentaje_hora = (
    (df_limpio["hora"] == HORA_POR_DEFECTO).sum() / len(df_limpio) * 100
)
print(
    f"El porcentaje de infracciones a la hora {HORA_POR_DEFECTO} "
    f"es {porcentaje_hora:.2f}%"
)


In [ ]:
push_si_corresponde()


## Punto 07

> Puntos: 1

Pendiente para un avance posterior: redacción de la conclusión final.
